# Light Attenuation (KD490)

Reprojects `KD490_2024.tif` to **EPSG:25833**, fills missing values inside the marine vanntyper AOI clipped to Møre og Romsdal (61.9°N – 63.5°N), and saves `KD490_2024_filled_25833.tif`.

In [1]:
from pathlib import Path
import subkart
import numpy as np
import rasterio as rio
import geoutils as gu
import xdem

In [2]:
niva_dir = Path("../niva")

out_path = subkart.light.fill_kd490(
    src_path=niva_dir / "KD490_2024.tif",
    out_path=niva_dir / "KD490_2024_filled_25833.tif",
)

Loading fill AOI...
  153 features, bounds: [ -54396.3908061  6873000.          296335.38999732 7052000.        ]
Reprojecting to EPSG:25833 at 1000 m...
  Shape: 354x181, NaN: 46875/64074
Rasterizing fill AOI mask...
Fine fill (maxSearchDist=200)...
0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.
Coarse fill (downsample 20x, maxSearchDist=500)...
0...10...20...30...40...50...60...70...80...90...100 - done.
Merging and applying final AOI mask...
0...10...20...30...40...50...60...70...80...90...100 - done.
Saved: ../niva/KD490_2024_filled_25833.tif


## Depth from DEM50 with sea map fallback


In [3]:

res = subkart.features.RESOLUTION  # 50 m

print("Loading sea map basisdata for Møre og Romsdal...")
gdf_sea = subkart.sources.sea_map_basisdata(["More_og_Romsdal"])
crs = gdf_sea.crs
gdf_sea = subkart.features.depth_preprocess(gdf_sea)

transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea, res=res)
print(f"Grid: {out_shape}, bounds: {bounds}")

Loading sea map basisdata for Møre og Romsdal...
Grid: (3673, 4349), bounds: (np.float64(-25100.0), np.float64(6908450.0), np.float64(192350.0), np.float64(7092100.0))


In [4]:
# Rasterize sea_avg_depth onto the sea map grid
vec = gu.Vector(gdf_sea)
sea_depth_raster = subkart.features.rasterize_area(
    vec, gdf_sea["sea_avg_depth"], bounds, res
)
sea_avg_depth = np.ma.filled(sea_depth_raster.data, np.nan).astype(np.float32)
print(f"Sea avg depth — valid: {np.sum(np.isfinite(sea_avg_depth))} / {sea_avg_depth.size}")

Sea avg depth — valid: 4466533 / 15973877


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


In [5]:
# Load DEM50 and resample to the sea map grid
print("Loading and resampling DEM50...")
dem = subkart.sources.dem_data()
dem = dem.crop(bounds)
dem = subkart.utils.resample_dem(dem, out_shape, transform, crs)
depth_dem = np.ma.filled(dem.data, np.nan).astype(np.float32)

print(f"DEM valid: {np.sum(np.isfinite(depth_dem))} / {depth_dem.size}")
print(f"DEM NaN:   {np.sum(np.isnan(depth_dem))}")

Loading and resampling DEM50...
DEM valid: 5404352 / 15973877
DEM NaN:   10569525


In [6]:
# Fill DEM gaps with sea map depth (same pattern as features.build)
# sea_avg_depth is positive; DEM stores depth as negative values
depth_filled = np.where(np.isnan(depth_dem), -sea_avg_depth, depth_dem)

print(f"After fill — NaN: {np.sum(np.isnan(depth_filled))} / {depth_filled.size}")
print(f"Depth range: {np.nanmin(depth_filled):.1f} – {np.nanmax(depth_filled):.1f} m")

After fill — NaN: 8477511 / 15973877
Depth range: -1343.1 – -0.0 m


## Photic / aphotic zone

Resample the filled KD490 (1000 m) to the depth grid (50 m), then apply the Beer-Lambert 1% light depth threshold:

$$z_{photic} = \frac{\ln(100)}{K_{d490}} \approx \frac{4.605}{K_{d490}}$$

A pixel is **photic** (`1`) if `|depth| < z_photic`, **aphotic** (`0`) otherwise.

In [7]:
# Resample filled KD490 to the depth grid
with rio.open(out_path) as kd_src:
    kd490_resampled = np.full(out_shape, np.nan, dtype=np.float32)
    rio.warp.reproject(
        source=rio.band(kd_src, 1),
        destination=kd490_resampled,
        dst_transform=transform,
        dst_crs=crs,
        src_nodata=np.nan,
        dst_nodata=np.nan,
        resampling=rio.warp.Resampling.bilinear,
    )

print(f"KD490 resampled — valid: {np.sum(np.isfinite(kd490_resampled))} / {kd490_resampled.size}")

KD490 resampled — valid: 3703182 / 15973877


In [9]:
# Photic zone base depth [m]
photic_depth = np.log(100) / kd490_resampled  # ≈ 4.605 / Kd490

depth_abs = np.abs(depth_filled)
valid = np.isfinite(depth_abs) & np.isfinite(kd490_resampled) & (kd490_resampled > 0)

photic_zone = np.full(out_shape, np.nan, dtype=np.float32)
photic_zone[valid] = np.where(depth_abs[valid] < photic_depth[valid], 1.0, 0.0)

print(f"Photic  pixels: {int(np.nansum(photic_zone == 1))}")
print(f"Aphotic pixels: {int(np.nansum(photic_zone == 0))}")
print(f"NoData  pixels: {int(np.sum(np.isnan(photic_zone)))}")

Photic  pixels: 782290
Aphotic pixels: 2067062
NoData  pixels: 13124525


## Save photic zone raster

In [11]:
fname = subkart.utils.to_filename("nisjedata-fotisk-sone", "moere-og-romsdal", "2026", "25833") + ".tif"

# Convert to uint8: photic=1, aphotic=0, nodata=255
photic_zone_int = np.full(out_shape, 255, dtype=np.uint8)
photic_zone_int[photic_zone == 1.0] = 1
photic_zone_int[photic_zone == 0.0] = 0

with rio.open(
    fname, "w",
    driver="GTiff",
    height=out_shape[0], width=out_shape[1],
    count=1,
    dtype=np.uint8,
    crs=crs,
    transform=transform,
    nodata=255,
    compress="deflate",
    tiled=True,
) as dst:
    dst.write(photic_zone_int, 1)

print("Saved:", fname)

Saved: nisjedata-fotisk-sone_moere-og-romsdal_2026_25833.tif
